# 06 - Branch and Bound

Branch-and-bound is a general strategy for solving optimization problems by **case splitting** and **bounds**. For integer programming, it repeatedly splits a fractional LP relaxation into smaller subproblems. A tree records those splits, while LP bounds show which parts of the tree cannot contain a better integer solution.

## Learning objectives

By the end of this notebook, you should be able to:

1. define a subproblem, node, root, parent, child, leaf, subtree, and search tree;
2. distinguish a branching rule from a node-selection or search rule;
3. explain upper and lower bounds for a maximization integer program;
4. derive the floor/ceiling branching disjunction for a fractional variable;
5. apply the three fathoming rules: infeasibility, integrality, and bound;
6. trace a complete depth-first branch-and-bound run; and
7. explain why an exhausted tree certifies global optimality.

## Prerequisites and scope

We use LP relaxations and bounds from Notebook 01 and the integer-hull example from Notebook 05. Familiarity with rooted trees is **not** assumed; all tree terminology is introduced below. We focus on linear **maximization** problems. A later section states how the bound directions change for minimization.

We consider

$$
\begin{aligned}
\max \quad & c^\top x \\
\text{subject to}\quad & Ax\le b,\\
&x_i\in\mathbb{Z} && i\in I,
\end{aligned}
$$

where variables outside the index set $I$ may remain continuous. If $I=\{1,\ldots,n\}$, the problem is pure integer; otherwise it is mixed-integer.


## From one problem to several subproblems

A **subproblem** is the original optimization problem together with additional restrictions. For example, from a problem containing an integer variable $x_1$, we may create the two cases

$$
x_1\le2
\qquad\text{and}\qquad
x_1\ge3.
$$

If every feasible integer solution satisfies one of the cases, solving both subproblems is equivalent to solving the original problem. The benefit is that each subproblem has a smaller feasible region and may be easier to solve or bound.

Branch-and-bound recursively repeats this idea. It does not normally enumerate individual integer points. Instead, one subproblem can represent many candidate points, and a single bound can eliminate that whole group at once.


## What are nodes and a search tree?

A **node** is the algorithm's record for one subproblem. It stores or implies:

- the additional restrictions defining that subproblem;
- its location in the case-splitting history;
- the solution and value of its LP relaxation, once solved; and
- its status, such as pending, branched, infeasible, integral, or pruned by bound.

The node is not a point of the feasible region. It represents an entire optimization problem and, therefore, a region containing zero, one, or many integer-feasible points.

The recursive relationships form a rooted **search tree**:

- The **root node** represents the original problem with no branching restrictions.
- **Branching** at a node creates two or more **child nodes** by adding mutually exhaustive case restrictions.
- The node that was split is the children's **parent**.
- An edge in the tree records one case restriction.
- A node's **depth** is the number of branch decisions from the root; the root has depth $0$.
- A **leaf** is a currently created node with no children.
- A node's **subtree** consists of that node and all descendants created below it.

For a two-way split, the first levels look like this:

```text
root: original problem
├── left child:  original constraints + first case
│   ├── left-left grandchild:  ... + another case
│   └── left-right grandchild: ... + complementary case
└── right child: original constraints + complementary first case
```

The complete tree is not known in advance; the algorithm builds only the parts it needs. **Search** means choosing which pending node to process next. It is separate from **branching**, which means choosing how to split the selected subproblem.


## Open, processed, and fathomed nodes

During the algorithm, a node moves through a small life cycle:

1. An **open**, **active**, or **pending** node has been created but not yet fully processed.
2. Processing solves its relaxation and decides what to do next.
3. A node is **branched** if children are created from it.
4. A node is **fathomed** when no descendant needs to be explored. The words **closed** and **pruned** are also common; we use *pruned* especially when a bound discards a whole subtree.

A branched node is no longer pending, but its children may be pending. The algorithm terminates when no pending nodes remain. At that point every case has either been solved or safely discarded.


## Relaxations provide node bounds

Let $Q$ denote the continuous region for one node: the original LP constraints plus that node's branching restrictions. Dropping integrality gives the **node LP**

$$
z_{LP}(Q)=\max\{c^\top x\mid x\in Q\}.
$$

Every integer-feasible point represented by the node is also feasible for this LP. Therefore,

$$
z_{IP}(Q)\le z_{LP}(Q).
$$

For maximization, the node LP value is an **upper bound** on every integer solution in that node and in every descendant, because descendants only add restrictions.

The best integer-feasible solution found anywhere so far is the **incumbent solution**. Its value $z_{inc}$ is a **lower bound** on the global optimum: the unknown optimum is at least as good as a solution we have actually found.

Keep the directions straight:

$$
\underbrace{z_{inc}}_{\text{known feasible value}}
\le
\underbrace{z_{IP}}_{\text{unknown optimum}}
\le
\underbrace{\max_{Q\text{ open}}z_{LP}(Q)}_{\text{best remaining upper bound}}.
$$

Before any integer solution is found, there is no incumbent. Conceptually, for maximization, its value starts at $-\infty$.


## Running example

We reuse the rational polytope from Notebook 05:

$$
P=\left\{x\in\mathbb{R}^2\ \middle|\
\begin{aligned}
x_1&\ge0,\\
x_2&\ge0,\\
2x_1+x_2&\le\frac{17}{2},\\
x_1+2x_2&\le\frac{17}{2}
\end{aligned}
\right\}.
$$

The integer program is

$$
\max\{x_1+x_2\mid x\in P\cap\mathbb{Z}^2\}.
$$

The **root node** represents this original model. Its relaxation is obtained by replacing $x\in\mathbb{Z}^2$ with $x\in\mathbb{R}^2$ while retaining all linear inequalities.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linprog

from src import hrep_to_vertices_2d, lattice_points_in_polytope_2d

A = np.array(
    [
        [-1.0, 0.0],       # x1 >= 0
        [0.0, -1.0],       # x2 >= 0
        [2.0, 1.0],        # 2*x1 + x2 <= 17/2
        [1.0, 2.0],        # x1 + 2*x2 <= 17/2
    ]
)
b = np.array([0.0, 0.0, 8.5, 8.5])
c = np.array([1.0, 1.0])
free_bounds = [(None, None)] * 2

root_relaxation = linprog(-c, A_ub=A, b_ub=b, bounds=free_bounds, method="highs")
root_x = root_relaxation.x
root_bound = -root_relaxation.fun

vertices = hrep_to_vertices_2d(A, b)
lattice_points = lattice_points_in_polytope_2d(A, b)
closed_vertices = np.vstack([vertices, vertices[0]])

fig, ax = plt.subplots(figsize=(7, 6))
ax.fill(
    closed_vertices[:, 0],
    closed_vertices[:, 1],
    color="tab:blue",
    alpha=0.15,
    label="$P$ (root LP relaxation)",
)
ax.plot(closed_vertices[:, 0], closed_vertices[:, 1], color="tab:blue")
ax.scatter(
    lattice_points[:, 0],
    lattice_points[:, 1],
    color="black",
    s=25,
    zorder=3,
    label=r"$P\cap\mathbb{Z}^2$",
)
ax.scatter(
    *root_x,
    marker="x",
    s=120,
    linewidths=2,
    color="tab:red",
    zorder=4,
    label="root LP optimum",
)
objective_x = np.linspace(0, 5, 100)
ax.plot(
    objective_x,
    root_bound - objective_x,
    linestyle="--",
    color="tab:red",
    label=rf"$x_1+x_2={root_bound:.2f}$",
)
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("The root relaxation has a fractional optimum")
ax.grid(True)
ax.legend()
plt.show()

print("Root LP solution:", root_x)
print("Root LP upper bound:", root_bound)

Both slanted constraints are tight at the root LP optimum:

$$
x^{LP}=\left(\frac{17}{6},\frac{17}{6}\right),
\qquad
z_{LP}=\frac{17}{3}\approx5.67.
$$

This value is a valid global upper bound. The point itself is not integer-feasible, so it cannot become the incumbent. The root is neither solved nor safely discardable; it must be branched.


## Branching on a fractional variable

Suppose an integer-constrained component has fractional LP value $x_j^*$. Every integer value of $x_j$ satisfies exactly one side of the disjunction

$$
\boxed{
x_j\le\lfloor x_j^*\rfloor
\qquad\text{or}\qquad
x_j\ge\lceil x_j^*\rceil.
}
$$

Here $\lfloor t\rfloor$ is the greatest integer at most $t$, and $\lceil t\rceil$ is the smallest integer at least $t$. The open interval between the floor and ceiling contains no integer. Consequently:

1. every integer-feasible point of the parent belongs to at least one child;
2. neither child contains the parent's fractional LP solution; and
3. each child relaxation is a subset of the parent relaxation, so its maximization bound cannot increase.

At the root, choose $x_1^*=17/6$. The two children represent

$$
Q_L=P\cap\{x_1\le2\},
\qquad
Q_R=P\cap\{x_1\ge3\}.
$$

The edge from the root to $Q_L$ records $x_1\le2$; the edge to $Q_R$ records $x_1\ge3$. The two child **integer** feasible sets partition the parent's integer points, even though the union of their continuous relaxations omits the fractional strip $2<x_1<3$.


In [ ]:
def solve_relaxation(variable_bounds):
    # This helper is deliberately specific to the running example.
    result = linprog(-c, A_ub=A, b_ub=b, bounds=variable_bounds, method="highs")
    return result.x, -result.fun


left_x, left_bound = solve_relaxation([(None, 2.0), (None, None)])
right_x, right_bound = solve_relaxation([(3.0, None), (None, None)])

print(f"Left child:  x = {left_x}, upper bound = {left_bound:.2f}")
print(f"Right child: x = {right_x}, upper bound = {right_bound:.2f}")

The child LP optima are

$$
\begin{aligned}
x^L&=\left(2,\frac{13}{4}\right),
&z_{LP}(Q_L)&=\frac{21}{4}=5.25,\\
x^R&=\left(3,\frac52\right),
&z_{LP}(Q_R)&=\frac{11}{2}=5.5.
\end{aligned}
$$

Both bounds improve on the root bound, but both LP solutions remain fractional. An LP bound belongs to a **subproblem**; it is not necessarily the value of an integer-feasible solution.


In [ ]:
def close_polygon(points):
    return np.vstack([points, points[0]])


left_A = np.vstack([A, [1.0, 0.0]])
left_b = np.append(b, 2.0)
right_A = np.vstack([A, [-1.0, 0.0]])
right_b = np.append(b, -3.0)

child_data = [
    (left_A, left_b, left_x, r"$Q_L$: $x_1\leq 2$"),
    (right_A, right_b, right_x, r"$Q_R$: $x_1\geq 3$"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (child_A, child_b, optimum, title) in zip(axes, child_data, strict=True):
    child_vertices = close_polygon(hrep_to_vertices_2d(child_A, child_b))
    ax.fill(
        closed_vertices[:, 0],
        closed_vertices[:, 1],
        color="tab:blue",
        alpha=0.06,
        label="parent relaxation",
    )
    ax.plot(closed_vertices[:, 0], closed_vertices[:, 1], color="tab:blue", alpha=0.3)
    ax.fill(
        child_vertices[:, 0],
        child_vertices[:, 1],
        color="tab:orange",
        alpha=0.25,
        label="child relaxation",
    )
    ax.plot(child_vertices[:, 0], child_vertices[:, 1], color="tab:orange")
    ax.scatter(lattice_points[:, 0], lattice_points[:, 1], color="black", s=20, zorder=3)
    ax.scatter(*optimum, marker="x", s=100, linewidths=2, color="tab:red", zorder=4)
    ax.set_xlim(-0.5, 5.0)
    ax.set_ylim(-0.5, 5.0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("$x_1$")
    ax.set_title(title)
    ax.grid(True)
axes[0].set_ylabel("$x_2$")
axes[0].legend(loc="upper left")
fig.suptitle("The branching disjunction partitions the integer points")
fig.tight_layout()
plt.show()

## The three ways to fathom a node

After solving a node LP, branch-and-bound can finish that node in three cases.

### 1. Infeasibility

If the LP relaxation is infeasible, the smaller integer-feasible set is also empty. The node and its entire potential subtree can be discarded.

### 2. Integrality

If the LP optimum satisfies every required integrality condition, it is feasible for the integer subproblem. Because it already maximizes over the larger LP set, no integer point in that node can do better. The solution may replace the incumbent, and the node needs no children.

### 3. Bound

If an incumbent exists and

$$
z_{LP}(Q)\le z_{inc},
$$

then even the node's optimistic LP bound cannot improve the incumbent. Every descendant has an equal or smaller upper bound, so the whole subtree is pruned.

Only a node that is LP-feasible, fractional in at least one integer-constrained variable, and capable of beating the incumbent needs to be branched. The order of the first two checks may vary in implementations, but their logical meaning is the same.


## Finding an incumbent by depth-first search

A **node-selection rule** chooses the next pending node. We begin with **depth-first search**: after branching, process one child and continue down that path before returning to siblings. A stack data structure implements this behavior.

Process $Q_L$ first. Its optimum has $x_2^L=13/4$, so branch again:

$$
Q_{LL}=Q_L\cap\{x_2\le3\},
\qquad
Q_{LR}=Q_L\cap\{x_2\ge4\}.
$$

The first grandchild has integer LP optimum $(2,3)$ with value $5$. It becomes the first incumbent. The sibling has LP bound $9/2=4.5$, so it cannot beat the incumbent and is pruned by bound. One feasible solution has eliminated a complete sibling subproblem without enumerating its integer points.


In [ ]:
left_left_x, left_left_bound = solve_relaxation([(None, 2.0), (None, 3.0)])
left_right_x, left_right_bound = solve_relaxation([(None, 2.0), (4.0, None)])

print(f"x1 <= 2, x2 <= 3: x = {left_left_x}, value = {left_left_bound:.2f}")
print(f"x1 <= 2, x2 >= 4: x = {left_right_x}, bound = {left_right_bound:.2f}")
print("Incumbent value:", left_left_bound)
print("Prune the second node by bound:", left_right_bound <= left_left_bound)

## Branching rules versus search rules

Two decisions are easy to confuse:

- A **branching rule** chooses a fractional variable or another disjunction used to create children. The simple rule here chooses the first fractional integer-constrained variable.
- A **node-selection rule** chooses which pending node to process. Depth-first search chooses a deepest recent node; **best-bound search** chooses the open node with the most promising LP bound.

Valid branching ensures correctness because no integer solutions are lost. Node selection affects efficiency and when good incumbents are found, but not the final answer if the finite search is completed. Practical solvers use sophisticated hybrids of these choices.


## The complete branch-and-bound algorithm

A basic maximization algorithm is:

1. Create the root node and place it on the pending-node list.
2. Start with no incumbent, so $z_{inc}=-\infty$.
3. While a pending node exists:
   1. select and remove one pending node;
   2. solve its LP relaxation;
   3. fathom it if the LP is infeasible;
   4. fathom it by bound if an incumbent exists and $z_{LP}\le z_{inc}$;
   5. if the LP solution satisfies integrality, update the incumbent if it is better, then fathom the node;
   6. otherwise choose a fractional integer-constrained variable, create the floor and ceiling children, and add them to the pending list.
4. If an incumbent exists when the pending list is empty, return it as optimal. If none exists, the integer problem is infeasible.

A compact pseudocode view is:

```text
pending <- {root}
incumbent <- none
while pending is not empty:
    Q <- select(pending)
    solve LP(Q)
    if LP(Q) is infeasible:                 fathom Q
    else if bound(Q) cannot beat incumbent: fathom Q
    else if LP solution is integer-feasible:
        update incumbent
        fathom Q
    else:
        branch Q and add its children to pending
return incumbent
```

The root, children, pending list, and fathoming rules now have precise algorithmic meanings rather than being merely a diagram.


## Educational implementation

The implementation in `src/optimization/branch_and_bound.py` handles small pure and mixed-integer maximization problems in inequality form. It uses depth-first node selection and first-fractional-variable branching. The returned node trace makes every decision inspectable.

```{literalinclude} ../src/optimization/branch_and_bound.py
:language: python
:linenos:
:start-after: "# --- BRANCH_AND_BOUND:solve:start"
:end-before: "# --- BRANCH_AND_BOUND:solve:end"
```

This solver is intended to expose the logic, not to compete with production MIP software.


In [ ]:
from src.optimization.branch_and_bound import branch_and_bound

result = branch_and_bound(c, A, b, bounds=free_bounds)

print("Status:", result.status)
print("Optimal integer solution:", result.x)
print("Optimal objective value:", result.objective)
print("Processed nodes:", len(result.nodes))

The solver returns $(2,3)$ with objective value $5$. The symmetric point $(3,2)$ is also optimal. Branch-and-bound must certify the **optimal value**, but it need not enumerate all solutions attaining that value.

The number of processed nodes counts subproblems whose LP relaxation was considered. It is not the number of feasible integer points.


## Reading the node trace

Each trace row corresponds to one processed node:

- `node` is a unique identifier;
- `parent` identifies the node whose branch created it;
- `depth` counts branch decisions from the root;
- `LP solution` and `LP bound` summarize the relaxation;
- `action` records why processing stopped or whether children were created;
- `branch variable` records the selected fractional component; and
- `incumbent` is the best feasible value known after that node was processed.

Identifiers encode creation order, not necessarily processing order. In depth-first search, both siblings can be created before the left subtree is fully processed.


In [ ]:
def format_point(point):
    if point is None:
        return "—"
    return np.array2string(point, precision=2, suppress_small=True)


trace = pd.DataFrame(
    [
        {
            "node": node.node_id,
            "parent": "—" if node.parent_id is None else node.parent_id,
            "depth": node.depth,
            "LP solution": format_point(node.lp_solution),
            "LP bound": "—" if node.lp_bound is None else f"{node.lp_bound:.2f}",
            "action": node.action,
            "branch variable": (
                "—" if node.branch_variable is None else f"x{node.branch_variable + 1}"
            ),
            "incumbent": (
                "—" if node.incumbent_value is None else f"{node.incumbent_value:.2f}"
            ),
        }
        for node in result.nodes
    ]
)
trace

In this run:

- node 3 supplies the incumbent value $5$;
- nodes 4, 7, and 8 are discarded because their LP bounds are at most $5$;
- node 6 is infeasible; and
- every other processed node is either branched or has an integer-feasible LP optimum.

The trace is a checkable certificate of the control flow: each terminal action corresponds to one of the three fathoming rules.


## Visualizing the complete search tree

The next cell reconstructs parent–child edges from the trace. Each box is one subproblem, not one feasible point. Blue boxes were branched; green boxes produced an integer optimum; orange boxes were pruned by bound; and red boxes were LP-infeasible.


In [ ]:
nodes_by_id = {node.node_id: node for node in result.nodes}
children = {node_id: [] for node_id in nodes_by_id}
for node in result.nodes:
    if node.parent_id is not None:
        children[node.parent_id].append(node.node_id)

leaf_counter = iter(range(sum(not child_ids for child_ids in children.values())))
x_positions = {}


def assign_x_position(node_id):
    if not children[node_id]:
        x_positions[node_id] = next(leaf_counter)
    else:
        child_positions = [assign_x_position(child_id) for child_id in children[node_id]]
        x_positions[node_id] = sum(child_positions) / len(child_positions)
    return x_positions[node_id]


assign_x_position(0)
colors = {
    "branched": "tab:blue",
    "integer": "tab:green",
    "bound": "tab:orange",
    "infeasible": "tab:red",
}

fig, ax = plt.subplots(figsize=(12, 6))
for node in result.nodes:
    x = x_positions[node.node_id]
    y = -node.depth
    if node.parent_id is not None:
        parent = nodes_by_id[node.parent_id]
        ax.plot(
            [x_positions[parent.node_id], x],
            [-parent.depth, y],
            color="0.55",
            zorder=1,
        )
    bound_label = "infeasible" if node.lp_bound is None else f"$z_{{LP}}={node.lp_bound:.2f}$"
    branch_label = (
        "" if node.branch_variable is None else f"\nbranch on $x_{node.branch_variable + 1}$"
    )
    label = f"N{node.node_id}\n{bound_label}\n{node.action}{branch_label}"
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=9,
        bbox={
            "boxstyle": "round,pad=0.35",
            "facecolor": colors[node.action],
            "alpha": 0.18,
            "edgecolor": colors[node.action],
        },
        zorder=2,
    )

ax.set_xlim(-0.7, max(x_positions.values()) + 0.7)
ax.set_ylim(-max(node.depth for node in result.nodes) - 0.6, 0.6)
ax.set_title("Complete depth-first branch-and-bound tree")
ax.axis("off")
plt.show()

The finished diagram is the **explored search tree**. A pruned node is drawn as a leaf because its uncreated descendants are unnecessary. This is one source of branch-and-bound's power: a small explored tree can account for a much larger implicit set of cases.


## Why the final incumbent is globally optimal

The proof rests on three facts.

1. **Branching is exhaustive for integer points.** The child disjunction preserves every integer-feasible point represented by the parent.
2. **LP bounds are valid.** A node's integer-feasible set lies inside its LP relaxation, and every descendant only adds restrictions.
3. **Fathoming is safe.** Infeasible nodes contain no candidates; an integral LP optimum solves its node; and a bound-pruned node cannot improve the incumbent.

When no pending node remains, every integer-feasible solution is accounted for by some fathomed node. None can have value greater than the incumbent, while the incumbent itself is feasible. Thus its value is the global optimum.

For a bounded pure-integer problem, there are only finitely many candidate lattice points. Repeated valid variable branching can eventually separate them, although the resulting tree may be exponentially large. More general convergence statements require care about boundedness, mixed-integer variables, tolerances, and branching rules.


## Global bounds, optimality gaps, and early stopping

At any time after finding an incumbent, a maximization search has

$$
z_{inc}\le z_{IP}\le U,
$$

where $U$ is the largest valid bound among open nodes. The absolute difference $U-z_{inc}$ is an **optimality gap**. If it is zero, optimality is proved even if some open nodes remain; each remaining node is then bound-prunable.

Production solvers often stop when an absolute or relative gap is below a user tolerance, or when a time or node limit is reached. In that case, the solver may return a feasible incumbent and a bound rather than a proof of exact optimality. Reporting both values tells the user what is known.


## What changes for minimization?

For minimization, an LP relaxation gives a **lower bound**, while a feasible incumbent gives an **upper bound**. A node can be pruned when its lower bound is no better than the incumbent:

$$
z_{LP}(Q)\ge z_{inc}.
$$

| problem sense | node LP bound | incumbent bound | prune by bound when |
|---|---|---|---|
| maximize | upper | lower | $z_{LP}(Q)\le z_{inc}$ |
| minimize | lower | upper | $z_{LP}(Q)\ge z_{inc}$ |

The branching disjunction and tree terminology do not change. Only the interpretation and comparison direction of objective bounds change.


## Mixed-integer variables

For a mixed-integer problem, only indices in $I$ are tested for integrality and eligible for integer branching. A continuous variable may remain fractional without preventing a node LP solution from being feasible for the MIP.

The helper accepts zero-based integer indices. For example,

```python
result = branch_and_bound(c, A, b, integer_indices=[0, 2])
```

requires mathematical variables $x_1$ and $x_3$ to be integral and leaves other variables continuous. Omitting the argument makes every variable integer-constrained.


## What practical solvers add

The core algorithm is simple; performance depends on strengthening nearly every step. Production solvers add, among other techniques:

- **presolve** and **bound propagation** to simplify models;
- **primal heuristics** to find incumbents early;
- **cutting planes** to tighten node relaxations, yielding branch-and-cut;
- **strong branching** and **pseudocosts** to choose variables;
- best-bound, depth-first, and hybrid node-selection strategies;
- reusable LP bases and warm starts;
- conflict analysis and domain propagation;
- numerical safeguards, parallel search, and resource limits.

These methods aim to improve bounds, find good feasible solutions, or avoid unnecessary nodes. The search tree can still be exponentially large in the worst case, which is consistent with the difficulty of general integer programming.


## Check your understanding

1. Does a node represent one point or one subproblem?
2. What is the difference between branching and node selection?
3. Why is an LP value an upper bound at a maximization node?
4. If $x_j^*=4.3$, what two variable branches are created?
5. Why may a node with LP bound $8$ be pruned when the incumbent value is $9$ in maximization?
6. What does it mean when the pending-node list is empty?

Brief answers: (1) a subproblem, potentially containing many points; (2) branching creates child cases, while node selection chooses which open case to process; (3) the LP contains every integer-feasible point of that node; (4) $x_j\le4$ and $x_j\ge5$; (5) no solution in that node can exceed its upper bound $8$; (6) every created case has been solved, branched, or safely fathomed, so the incumbent is optimal if one exists.


## Takeaways and further reading

- A node is a subproblem; parent–child relationships between branch decisions form a search tree.
- The root is the original problem, and descendants add restrictions.
- For maximization, node LPs provide upper bounds and an integer incumbent provides a lower bound.
- Fractional-variable branching uses an exhaustive floor/ceiling disjunction.
- Nodes are fathomed by infeasibility, integrality, or a bound that cannot beat the incumbent.
- Branching rules create cases; node-selection rules determine the order of search.
- An exhausted and safely fathomed tree is a certificate of global optimality.

For a solver-oriented introduction with examples, see the open [SCIP Optimization Suite book](https://scipbook.readthedocs.io/en/latest/). The [COIN-OR branch-and-cut overview](https://www.coin-or.org/ABACUS/coin.html) gives additional historical and algorithmic context. Production behavior and terminology are documented in the [SCIP reference manual](https://www.scipopt.org/doc/html/).


## Next

Branch-and-bound may be unnecessary when an LP relaxation already has an integral optimum. The next notebook studies structural conditions—especially total unimodularity and total dual integrality—that explain when this happens.

➡️ [07 - Unimodularity and Total Dual Integrality](07-Unimodularity-and-Total-Dual-Integrality.ipynb)
